In [1]:
import cobra
import os
import pickle
import re
from cobra import Reaction, Metabolite
import requests
import copy

pardir= os.path.join(os.getcwd(), os.pardir)
cow_before_path= os.path.join(pardir,'data/change_gene_name_cow.xml')
cow= cobra.io.read_sbml_model(cow_before_path)
cow_before= copy.deepcopy(cow)
cow_metabolite_dict= dict()
for metabolite in cow_before.metabolites:
    if 'kegg.compound' in metabolite.annotation:
        if not metabolite.annotation['kegg.compound'] in cow_metabolite_dict:
            cow_metabolite_dict[metabolite.annotation['kegg.compound']]= [metabolite.id]
        else:
            before= cow_metabolite_dict[metabolite.annotation['kegg.compound']]
            before.append(metabolite.id)
            cow_metabolite_dict[metabolite.annotation['kegg.compound']]= before
            

In [2]:
with open(os.path.join(pardir,'data/gene_reaction.pickle'),'rb') as f:
    gene_reaction= pickle.load(f)

with open(os.path.join(pardir,'data/metabolite_add.pickle'),'rb') as f:
    metabolite_add= pickle.load(f)

with open(os.path.join(pardir,'data/metabolite_formula.pickle'),'rb') as f:
    metabolite_formula= pickle.load(f)

with open(os.path.join(pardir,'data/metabolite_overlap.pickle'),'rb') as f:
    metabolite_overlap= pickle.load(f)

with open(os.path.join(pardir,'data/reaction_add.pickle'),'rb') as f:
    reaction_add= pickle.load(f)
          
with open(os.path.join(pardir,'data/reaction_overlap.pickle'),'rb') as f:
    reaction_overlap= pickle.load(f)

In [3]:
pattern = r'[C,G]([0-9]*)'
total_metabolite_related_reaction_add= set()
for kegg_id, metabolite_total in reaction_add.items():
    for reactant in metabolite_total[0]:
        a= re.search(pattern,reactant)
        reactant_name= a.group()
        total_metabolite_related_reaction_add.add(reactant_name)
    for product in metabolite_total[1]:
        a= re.search(pattern,product)
        product_name= a.group()
        total_metabolite_related_reaction_add.add(product_name)

In [4]:
#cytosol 없는 metabolite 더하기 cytosol metabolite가 없을 경우
metabolite_add_list= list()
reaction_add_list= list()
for kegg_id, compartments in cow_metabolite_dict.items():
    if kegg_id in total_metabolite_related_reaction_add:
        metabolite_original= cow_before.metabolites.get_by_id(compartments[0])
        metabolite_id_= metabolite_original.id
        metabolite_name_= metabolite_original.name
        metabolite_formula_=metabolite_original.formula
        metabolite_annotation_= metabolite_original.annotation
        not_add_c_criteria= False
        for compartment in compartments:
            if 'c' == compartment[-1]:
                not_add_c_criteria= True
        if not not_add_c_criteria:
            c_metabolite = Metabolite(
                metabolite_id_[:-1]+'c',
                formula=metabolite_formula_,
                name=metabolite_name_,
                compartment='c')
            c_metabolite.annotation= metabolite_annotation_
            metabolite_add_list.append(c_metabolite)
cow_before.add_metabolites(metabolite_add_list)

#MAM00154r <-> MAM00154c 
reaction = Reaction('MAR13087')
reaction.name = 'Transport G00008 from r to c'
reaction.subsystem = 'Transport reactions'
reaction.lower_bound = -1000.
reaction.upper_bound = 1000.
reaction.add_metabolites({
    cow_before.metabolites.get_by_id('MAM00154r'): -1.0,
    cow_before.metabolites.get_by_id('MAM00154c'): 1.0
})
reaction_add_list.append(reaction)

#MAM00154c <-> sink reaction
reaction = Reaction('MAR13088')
reaction.name = 'Demand reaction of G00008'
reaction.subsystem = 'Exchange/demand reactions'
reaction.lower_bound = 0.
reaction.upper_bound = 1000.
reaction.add_metabolites({
    cow_before.metabolites.get_by_id('MAM00154c'): -1.0,
})
reaction_add_list.append(reaction)

#MAM02321m <-> MAM02321c, MAM02321c <-> 
reaction = Reaction('MAR13089')
reaction.name = 'Transport L-2-amino-3-oxobutanoic acid from m to c'
reaction.subsystem = 'Transport reactions'
reaction.lower_bound = -1000.
reaction.upper_bound = 1000.
reaction.add_metabolites({
    cow_before.metabolites.get_by_id('MAM02321m'): -1.0,
    cow_before.metabolites.get_by_id('MAM02321c'): 1.0
})
reaction_add_list.append(reaction)

#MAM00154c <-> sink reaction
reaction = Reaction('MAR13090')
reaction.name = 'Demand reaction of L-2-amino-3-oxobutanoic acid'
reaction.subsystem = 'Exchange/demand reactions'
reaction.lower_bound = 0.
reaction.upper_bound = 1000.
reaction.add_metabolites({
    cow_before.metabolites.get_by_id('MAM02321c'): -1.0,
})
reaction_add_list.append(reaction)

cow_before.add_reactions(reaction_add_list)

In [5]:
#실제 kegg id가 원래 human metabolite에 없는 metabolite 더하기
#metabolite과 관련된 transport reaction, exchange reaction 더하기
metabolite_id_number_start= max([int(metabolite.id[:-1].replace('MAM','')) for reaction in cow_before.reactions])+1
reaction_id_number_start= max([int(reaction.id.replace('MAR','')) for reaction in cow_before.reactions])+1
metabolite_add_list= list()
reaction_add_list= list()

for metabolite in metabolite_add:
    #cytosol add
    url= 'https://rest.kegg.jp/get/'+metabolite
    requestData = requests.get(url).text
    for data in requestData.split('\n'):
        if 'NAME' in data:
            metabolite_name= data.split('NAME        ')[1].replace(';','')
    metabolite_id_= 'MAM'+str(metabolite_id_number_start).zfill(5)
    metabolite_formula_=metabolite_formula[metabolite]
    metabolite_annotation= {'kegg.compound': metabolite} 
    c_metabolite = Metabolite(
                metabolite_id_+'c',
                formula=metabolite_formula_,
                name=metabolite_name,
                compartment='c')
    c_metabolite.annotation= metabolite_annotation
    
    e_metabolite = Metabolite(
                metabolite_id_+'e',
                formula=metabolite_formula_,
                name=metabolite_name,
                compartment='e')
    e_metabolite.annotation= metabolite_annotation
    metabolite_add_list.append(c_metabolite)
    metabolite_add_list.append(e_metabolite)
    metabolite_id_number_start+=1
    
    #transport reaction
    reaction_id= 'MAR'+ str(reaction_id_number_start).zfill(5)
    reaction = Reaction(reaction_id)
    reaction.name = 'Transport '+ metabolite_name+ ' from c to e'
    reaction.subsystem = 'Transport reactions'
    reaction.lower_bound = -1000.
    reaction.upper_bound = 1000.
    reaction.add_metabolites({
        c_metabolite: -1.0,
        e_metabolite: 1.0
    })
    reaction_add_list.append(reaction)
    reaction_id_number_start+=1
    
    #exchange reaction
    reaction_id= 'MAR'+ str(reaction_id_number_start).zfill(5)
    reaction = Reaction(reaction_id)
    reaction.name = 'Exchange reaction of '+ metabolite_name
    reaction.subsystem = 'Exchange/demand reactions'
    reaction.lower_bound = -1000.
    reaction.upper_bound = 1000.
    reaction.add_metabolites({
        e_metabolite: -1.0,
    })
    reaction_add_list.append(reaction)
    reaction_id_number_start+=1
    
cow_before.add_metabolites(metabolite_add_list)
cow_before.add_reactions(reaction_add_list)

In [6]:
cow_metabolite_dict_revise= dict()
for metabolite in cow_before.metabolites:
    if 'kegg.compound' in metabolite.annotation and metabolite.compartment=='c':
        cow_metabolite_dict_revise[metabolite.annotation['kegg.compound']]= [metabolite.id]

In [7]:
reaction_add_list= list()
reaction_id_number= max_reaction_id= max([int(reaction.id.replace('MAR','')) for reaction in cow_before.reactions])+1
for kegg_id, metabolite_list in reaction_add.items():
    reactant_list= metabolite_list[0]
    product_list= metabolite_list[1]
    reaction_coeff_dict= dict()
    for reactant in reactant_list:
        a= re.search(pattern,reactant)
        metabolite_name= a.group()
        metabolite_coeff= reactant.replace(metabolite_name,'').strip()
        if metabolite_coeff=='':
            metabolite_coeff= 1
        else:
            metabolite_coeff= int(metabolite_coeff)
        #metabolite_overlap에 있을경우
        if metabolite_name in metabolite_overlap:
            metabolite_name= metabolite_overlap[metabolite_name]
        corresponding_metabolite= cow_metabolite_dict_revise[metabolite_name][0]
        reaction_coeff_dict[cow_before.metabolites.get_by_id(corresponding_metabolite)]= -metabolite_coeff
        
    for product in product_list:
        a= re.search(pattern,product)
        metabolite_name= a.group()
        metabolite_coeff= product.replace(metabolite_name,'').strip()
        if metabolite_coeff=='':
            metabolite_coeff= 1
        else:
            metabolite_coeff= int(metabolite_coeff)
        #metabolite_overlap에 있을경우
        if metabolite_name in metabolite_overlap:
            metabolite_name= metabolite_overlap[metabolite_name]
        corresponding_metabolite= cow_metabolite_dict_revise[metabolite_name][0]
        reaction_coeff_dict[cow_before.metabolites.get_by_id(corresponding_metabolite)]= metabolite_coeff
        
    url= 'https://rest.kegg.jp/get/'+kegg_id
    requestData = requests.get(url).text
    reaction_name= ''
    subsystem= ''
    for data in requestData.split('\n'):
        if 'NAME' in data:
            reaction_name= data.split('NAME        ')[1].replace(';','')
        if 'PATHWAY' in data:
            subsystem= data.split('PATHWAY     ')[1].split('  ')[1]
            
    reaction_id= 'MAR'+str(reaction_id_number).zfill(5)
    reaction = Reaction(reaction_id)
    reaction.add_metabolites(reaction_coeff_dict)
    reaction.name = reaction_name
    reaction.subsystem = subsystem
    reaction.lower_bound = -1000.  # This is the default
    reaction.upper_bound = 1000.
    reaction_id_number+=1
    reaction.annotation= {'kegg.reaction': kegg_id}
    reaction_add_list.append(reaction)
cow_before.add_reactions(reaction_add_list)

In [8]:
reaction_gene= dict()
for gene, reaction_list in gene_reaction.items():
    for reaction in reaction_list:
        if not reaction in reaction_gene:
            reaction_gene[reaction]=[gene]
        else:
            reaction_gene[reaction]=reaction_gene[reaction]+[gene]

In [9]:
#gpr rule에 gene 추가
for reaction in cow_before.reactions:
    if 'kegg.reaction' in reaction.annotation and reaction.annotation['kegg.reaction'] in reaction_gene:
        gene_add_list = reaction_gene[reaction.annotation['kegg.reaction']]
        original_gpr_rule= reaction.gene_reaction_rule
        for gene in gene_add_list:
            if gene in original_gpr_rule:
                continue
                
            if original_gpr_rule == '':
                original_gpr_rule+= gene
            else:
                original_gpr_rule= original_gpr_rule+ ' or '+ gene     
        reaction.gene_reaction_rule= original_gpr_rule

In [17]:
from cobra.io import write_sbml_model

write_sbml_model(cow_before, "cow_add_information.xml")